# How Chat Models Are Really Trained: Pretraining, SFT and DPO

The first GuppyLM notebook trained a model in one go. Real chat models go through **stages**. Each stage uses the same model and the same loop. Only the **data** and the **loss** change.

| Stage | Data | What the model learns | Status in this notebook |
|---|---|---|---|
| 0. Pretraining | Plain text, no chat format | The language itself: which words follow which | Done below |
| 1. SFT (supervised fine-tuning) | Chat conversations | To answer when asked (loss only on the answer) | Done below |
| 2. DPO (direct preference optimization) | Pairs of a good and a bad answer | To prefer the good one | Coming after SFT |

You will chat with the model after every stage, so you can see what each one adds.

**Honest limits.** The data is synthetic fish talk made from templates, so every stage here is a miniature of the real thing. Real pretraining uses trillions of tokens of web text. The mechanics are the same, the scale is not.

**How to run it.** Pick a GPU (**Runtime > Change runtime type**), then run the cells from top to bottom.

## 1. Setup

In [ ]:
!pip install -q tokenizers datasets

import os, sys, shutil, time, json, random
import torch

# If you run this notebook a second time in the same session, Python would keep the OLD copies of our
# modules in memory. Forget them, so the files written below are imported fresh.
for name in ['config', 'model', 'dataset', 'train', 'stages', 'inference']:
    sys.modules.pop(name, None)

# Start from an empty folder so old files from earlier runs cannot confuse us.
if os.path.exists('/content/guppy'):
    shutil.rmtree('/content/guppy')
os.makedirs('/content/guppy')
os.chdir('/content/guppy')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__}, device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('No GPU found. Training on CPU is about 20 times slower.')

## 2. The source files

These are the real GuppyLM files. `config.py`, `model.py`, `train.py` and `inference.py` are the same ones the first notebook used. `dataset.py` is only needed here because `train.py` imports it (and `stages.py` borrows `get_lr` from `train.py`).
The new file is **`stages.py`**. It holds everything the stages share: a packed-text dataset, loss functions, and one training loop that every stage reuses. Read its top comment first.

In [ ]:
%%writefile config.py
"""GuppyLM configuration."""

from dataclasses import dataclass


# A dataclass is a Python class that mostly holds data. Python writes __init__ for you,
# so GuppyConfig(n_layers=2) works. (Java comparison: a record, or a plain class with fields.)
@dataclass
class GuppyConfig:
    # Shape of the model. Change these and the model gets bigger or smaller.
    vocab_size: int = 4096    # how many different tokens the model knows (tokenizer may use fewer)
    max_seq_len: int = 128    # longest text, in tokens, the model reads at once
    d_model: int = 384        # width: every token is represented by a vector of this many numbers
    n_layers: int = 6         # how many transformer blocks are stacked on top of each other
    n_heads: int = 6          # attention heads per block; d_model must divide evenly by this
    ffn_hidden: int = 768     # inner width of the feed-forward network (2 x d_model here)
    dropout: float = 0.1      # during training, randomly zero 10% of values to reduce memorising

    # Special tokens. These ids must match the order given to the tokenizer trainer.
    pad_id: int = 0           # <pad>: filler so every sentence in a batch has the same length
    bos_id: int = 1           # <|im_start|>: a message begins
    eos_id: int = 2           # <|im_end|>: a message ends (the model learns to emit this to stop)


@dataclass
class TrainConfig:
    # How training runs. None of these change the model's shape.
    batch_size: int = 32            # sentences processed together in one step
    learning_rate: float = 3e-4     # peak size of each weight update (3e-4 = 0.0003)
    min_lr: float = 3e-5            # the learning rate falls to this at the end of training
    weight_decay: float = 0.1       # gently pulls weights toward 0, which helps against memorising
    warmup_steps: int = 200         # steps spent raising the learning rate from 0 to its peak
    max_steps: int = 10000          # total number of weight updates
    eval_interval: int = 200        # test on held-out data every this many steps
    save_interval: int = 500        # save a snapshot every this many steps
    grad_clip: float = 1.0          # cap on the total gradient size, so one bad batch cannot wreck the model
    device: str = "auto"            # "auto" picks cuda, then mps (Apple), then cpu
    seed: int = 42                  # fixes the random starting point so runs can be repeated
    data_dir: str = "data"          # folder with train.jsonl, eval.jsonl and tokenizer.json
    output_dir: str = "checkpoints" # folder where trained weights are saved

In [ ]:
%%writefile model.py
"""
GuppyLM — a tiny fish brain.

Vanilla transformer: multi-head attention, ReLU FFN, LayerNorm, learned positional embeddings.
No GQA, no SwiGLU, no parallel residual, no RoPE. As simple as it gets.
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from config import GuppyConfig


# Shape letters used in the comments below:
#   B = batch size (sentences at once), T = tokens per sentence, C = model width (d_model),
#   H = number of attention heads, D = size of one head (C / H).
class Attention(nn.Module):
    # Attention lets each token look at earlier tokens and mix in information from the ones that matter.
    def __init__(self, config):
        super().__init__()
        self.n_heads = config.n_heads
        self.head_dim = config.d_model // config.n_heads  # each head works on a slice of the vector

        # One Linear layer makes query, key and value together (3 x C outputs). Think of it as a lookup:
        #   query = "what am I looking for?", key = "what do I contain?", value = "what do I hand over?"
        self.qkv = nn.Linear(config.d_model, 3 * config.d_model)
        self.out = nn.Linear(config.d_model, config.d_model)  # mixes the heads' results back together
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x, mask=None):
        B, T, C = x.shape
        # (B, T, 3C) -> (B, T, 3, H, D) -> (3, B, H, T, D), so q, k and v each come out as (B, H, T, D).
        qkv = self.qkv(x).reshape(B, T, 3, self.n_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        # Score for every pair of tokens: how well does token i's query match token j's key?
        # Result shape (B, H, T, T). Dividing by sqrt(D) keeps the numbers small so softmax does not saturate.
        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None:
            # Where the mask is 0 (a future token), set the score to -infinity.
            # Softmax turns -infinity into probability 0, so a token cannot peek at what comes after it.
            attn = attn.masked_fill(mask == 0, float("-inf"))
        # Softmax turns each row of scores into probabilities that add up to 1 (the "attention weights").
        attn = self.dropout(F.softmax(attn, dim=-1))
        # Weighted average of the values, then put the heads side by side again: (B, H, T, D) -> (B, T, C).
        return self.out((attn @ v).transpose(1, 2).contiguous().view(B, T, C))


class FFN(nn.Module):
    # Feed-forward network: processes each token on its own. Attention mixes information BETWEEN tokens,
    # this part thinks about each token by itself. It widens the vector (C -> 768), applies ReLU, narrows it back.
    def __init__(self, config):
        super().__init__()
        self.up = nn.Linear(config.d_model, config.ffn_hidden)
        self.down = nn.Linear(config.ffn_hidden, config.d_model)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        # ReLU(x) = max(0, x). Without a non-linear step like this, stacked Linear layers would collapse into one.
        return self.dropout(self.down(F.relu(self.up(x))))


class Block(nn.Module):
    # One transformer block: attention, then FFN. The model stacks n_layers of these.
    def __init__(self, config):
        super().__init__()
        self.norm1 = nn.LayerNorm(config.d_model)
        self.attn = Attention(config)
        self.norm2 = nn.LayerNorm(config.d_model)
        self.ffn = FFN(config)

    def forward(self, x, mask=None):
        # "x = x + ..." is a residual connection: the block adds a correction to its input instead of
        # replacing it. That keeps the original signal intact and makes deep stacks trainable.
        # LayerNorm rescales each token's vector to a steady size before each sub-layer.
        x = x + self.attn(self.norm1(x), mask)
        x = x + self.ffn(self.norm2(x))
        return x


class GuppyLM(nn.Module):
    def __init__(self, config: GuppyConfig):
        super().__init__()
        self.config = config

        # An Embedding is a table: token id -> vector of C numbers. The numbers are learned in training.
        self.tok_emb = nn.Embedding(config.vocab_size, config.d_model)
        # Attention itself has no sense of order, so we also add a learned vector for each POSITION (0, 1, 2, ...).
        self.pos_emb = nn.Embedding(config.max_seq_len, config.d_model)
        self.drop = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([Block(config) for _ in range(config.n_layers)])
        self.norm = nn.LayerNorm(config.d_model)
        # The last layer turns a vector of C numbers into one score per vocabulary token ("logits").
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        # Weight tying: the output layer reuses the embedding table. It saves about 1.6M parameters and
        # makes sense, since "id -> vector" and "vector -> id" are mirror images of each other.
        self.lm_head.weight = self.tok_emb.weight  # tie weights

        self.apply(self._init_weights)  # call _init_weights on every sub-module

    def _init_weights(self, m):
        # Start from small random numbers (standard deviation 0.02) and zero biases.
        # Big random starting weights make early training unstable.
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        # idx: token ids, shape (B, T). targets: the "next token" answers, same shape (only used in training).
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)                  # 0, 1, 2, ..., T-1
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))      # (B, T, C): what the token is + where it is
        # Lower-triangular matrix of ones: row i has 1s only for columns 0..i (the tokens up to and including i).
        # The two unsqueeze calls add the batch and head dimensions so the mask fits (B, H, T, T) scores.
        mask = torch.tril(torch.ones(T, T, device=idx.device)).unsqueeze(0).unsqueeze(0)

        for block in self.blocks:
            x = block(x, mask)

        logits = self.lm_head(self.norm(x))                       # (B, T, vocab_size)

        loss = None
        if targets is not None:
            # Cross-entropy compares the predicted scores with the true next token. A perfect guess gives 0;
            # a random guess over 4096 tokens gives about ln(4096) = 8.3.
            # view(-1, ...) flattens (B, T) into one long list of predictions.
            # ignore_index=0 skips <pad> positions, so filler tokens are not counted.
            loss = F.cross_entropy(
                logits.view(-1, self.config.vocab_size),
                targets.view(-1),
                ignore_index=0,
            )

        return logits, loss

    # @torch.no_grad() turns off gradient tracking. We only generate here, not train, so it saves memory and time.
    @torch.no_grad()
    def generate(self, idx, max_new_tokens=64, temperature=0.7, top_k=50, **kwargs):
        self.eval()  # switch off dropout
        for _ in range(max_new_tokens):
            # The model can only read max_seq_len tokens, so keep just the most recent ones.
            idx_cond = idx[:, -self.config.max_seq_len:]
            # Run the whole sequence again for every new token. (Real systems cache earlier work, the "KV cache".
            # We skip that to keep the code simple.)
            logits, _ = self(idx_cond)
            # Only the LAST position matters: it holds the scores for the next token.
            # Dividing by temperature: below 1 makes the top choice even more likely, above 1 flattens the odds.
            logits = logits[:, -1, :] / temperature
            if top_k > 0:
                # Top-k sampling: keep only the k best-scoring tokens and rule out the rest, so a very
                # unlikely token cannot be picked by accident.
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")
            probs = F.softmax(logits, dim=-1)                       # scores -> probabilities
            next_id = torch.multinomial(probs, num_samples=1)       # draw one token at random using those odds
            idx = torch.cat([idx, next_id], dim=1)                  # append it and go around again
            # .item() only works when there is exactly one sentence (batch size 1), which is how we chat.
            if next_id.item() == self.config.eos_id:
                break                                               # the model wrote <|im_end|>: it is done
        return idx, []

    def param_count(self):
        # parameters() lists every learnable tensor. Tied weights are the same tensor, so they count once.
        total = sum(p.numel() for p in self.parameters())
        return total, 0

    def param_summary(self):
        total, _ = self.param_count()
        return f"GuppyLM: {total:,} params ({total/1e6:.1f}M)"

In [ ]:
%%writefile dataset.py
"""GuppyLM dataset loading."""

import json

import torch
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer


# A PyTorch Dataset answers two questions: "how many samples?" (__len__) and
# "give me sample number i" (__getitem__). The DataLoader below builds batches from it.
class GuppyDataset(Dataset):
    def __init__(self, path: str, tokenizer_path: str, max_len: int = 512):
        self.tokenizer = Tokenizer.from_file(tokenizer_path)
        self.max_len = max_len
        self.samples = []

        # Read the JSONL file (one JSON object per line) and turn every text into token ids once,
        # up front. All 57,000 samples stay in memory, which is fine because they are short.
        with open(path) as f:
            for line in f:
                data = json.loads(line)
                ids = self.tokenizer.encode(data["text"]).ids
                if len(ids) > max_len:
                    ids = ids[:max_len]        # cut anything longer than the model can read
                if len(ids) >= 2:              # need at least one input token and one target token
                    self.samples.append(ids)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        ids = self.samples[idx]
        # The whole training task is here. The target is the input shifted by one token:
        #   input  x: [A, B, C, D]
        #   target y: [B, C, D, E]
        # At each position the model must predict the NEXT token.
        x = ids[:-1]
        y = ids[1:]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)


def collate_fn(batch, pad_id=0):
    # Sentences have different lengths, but a tensor must be a rectangle.
    # So we pad every sentence in the batch with pad_id up to the longest one.
    # The loss ignores pad_id (see ignore_index=0 in model.py), so padding does not teach anything.
    xs, ys = zip(*batch)
    max_len = max(len(x) for x in xs)
    padded_x = torch.full((len(xs), max_len), pad_id, dtype=torch.long)
    padded_y = torch.full((len(ys), max_len), pad_id, dtype=torch.long)
    for i, (x, y) in enumerate(zip(xs, ys)):
        padded_x[i, :len(x)] = x
        padded_y[i, :len(y)] = y
    return padded_x, padded_y


def get_dataloader(path, tokenizer_path, max_len=512, batch_size=32, shuffle=True):
    dataset = GuppyDataset(path, tokenizer_path, max_len)
    # shuffle=True gives a new random order each epoch (one pass over the data) for training.
    # num_workers=0 loads batches in the main process: simple, and fast enough for tiny data.
    # pin_memory=True puts batches in page-locked memory, which makes copying to the GPU faster.
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle,
        collate_fn=collate_fn, num_workers=0, pin_memory=True,
    )

In [ ]:
%%writefile train.py
"""GuppyLM training loop."""

import json
import math
import os
import time

import torch

from config import GuppyConfig, TrainConfig
from dataset import get_dataloader
from model import GuppyLM


def get_device(config):
    # Pick where the math runs. A GPU is much faster than a CPU for this kind of work.
    if config.device == "auto":
        if torch.cuda.is_available():
            return torch.device("cuda")      # NVIDIA GPU (Colab)
        if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            return torch.device("mps")       # Apple Silicon GPU
        return torch.device("cpu")
    return torch.device(config.device)


def get_lr(step, config):
    # Learning rate schedule: linear warmup, then cosine decay.
    # Warmup: at the start the weights are random and the gradients are noisy, so take small steps first.
    if step < config.warmup_steps:
        return config.learning_rate * step / config.warmup_steps
    # progress runs from 0.0 (end of warmup) to 1.0 (last step).
    progress = (step - config.warmup_steps) / max(1, config.max_steps - config.warmup_steps)
    # coeff follows half a cosine wave: 1.0 at progress 0, falling smoothly to 0.0 at progress 1.
    coeff = 0.5 * (1 + math.cos(math.pi * progress))
    # Blend between the final rate (min_lr) and the peak rate (learning_rate) using coeff.
    return config.min_lr + (config.learning_rate - config.min_lr) * coeff


@torch.no_grad()  # no gradients needed when we only measure the loss
def evaluate(model, loader, device, max_batches=50):
    # Average loss on data the model has NOT trained on. If this stops improving while the training
    # loss keeps falling, the model is memorising instead of learning.
    model.eval()   # turns dropout off, so results are not random
    total_loss, n = 0, 0
    for x, y in loader:
        if n >= max_batches:   # 50 batches is enough for a stable estimate, and it keeps evaluation quick
            break
        x, y = x.to(device), y.to(device)
        _, loss = model(x, y)
        total_loss += loss.item()   # .item() turns a one-number tensor into a normal Python float
        n += 1
    model.train()  # back to training mode (dropout on)
    return total_loss / max(1, n)


def train():
    mc = GuppyConfig()   # model settings
    tc = TrainConfig()   # training settings
    device = get_device(tc)
    torch.manual_seed(tc.seed)   # same seed -> same random start -> repeatable runs

    print(f"Device: {device}")

    tokenizer_path = os.path.join(tc.data_dir, "tokenizer.json")
    model = GuppyLM(mc).to(device)   # .to(device) copies all weights onto the GPU
    print(model.param_summary())

    # Training data is shuffled every epoch. Eval data is not, so eval numbers are comparable between runs.
    train_loader = get_dataloader(
        os.path.join(tc.data_dir, "train.jsonl"), tokenizer_path,
        mc.max_seq_len, tc.batch_size, shuffle=True,
    )
    eval_loader = get_dataloader(
        os.path.join(tc.data_dir, "eval.jsonl"), tokenizer_path,
        mc.max_seq_len, tc.batch_size, shuffle=False,
    )
    print(f"Train: {len(train_loader.dataset):,}, Eval: {len(eval_loader.dataset):,}")

    # The optimizer decides how to change each weight, given its gradient. AdamW is the standard choice
    # for transformers: it adapts the step size per weight and applies weight decay properly.
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=tc.learning_rate,
        weight_decay=tc.weight_decay, betas=(0.9, 0.95),
    )

    # Mixed precision: on a GPU, run most math in 16-bit numbers. It is faster and uses less memory.
    # 16-bit gradients can be so small that they round to zero, so GradScaler multiplies the loss by a
    # large number before the backward pass and divides it back out afterwards.
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda") if use_amp else None

    os.makedirs(tc.output_dir, exist_ok=True)
    # Save the settings next to the weights, so inference can rebuild a model of the same shape.
    with open(os.path.join(tc.output_dir, "config.json"), "w") as f:
        json.dump({"model": vars(mc), "train": vars(tc)}, f, indent=2)

    model.train()   # turn dropout on
    step, best_eval = 0, float("inf")
    losses = []     # the loss of every step, kept so we can plot it later
    t0 = time.time()

    print(f"\nTraining for {tc.max_steps} steps...")
    print(f"{'Step':>6} | {'LR':>10} | {'Train':>10} | {'Eval':>10} | {'Time':>8}")
    print("-" * 56)

    # One "step" = one batch = one weight update. The outer loop restarts the data when it runs out
    # (a new epoch) and the inner break stops exactly at max_steps.
    while step < tc.max_steps:
        for x, y in train_loader:
            if step >= tc.max_steps:
                break

            x, y = x.to(device), y.to(device)
            # Set this step's learning rate from the schedule.
            lr = get_lr(step, tc)
            for pg in optimizer.param_groups:
                pg["lr"] = lr

            if use_amp:
                # GPU path, with mixed precision.
                with torch.amp.autocast("cuda"):
                    _, loss = model(x, y)                    # 1. forward pass: predictions and loss
                scaler.scale(loss).backward()                # 2. backward pass: a gradient for every weight
                scaler.unscale_(optimizer)                   # undo the loss scaling before we look at the gradients
                # Gradient clipping: if the total gradient is larger than grad_clip, shrink it to that size.
                torch.nn.utils.clip_grad_norm_(model.parameters(), tc.grad_clip)
                scaler.step(optimizer)                       # 3. update the weights (skipped if gradients overflowed)
                scaler.update()                              # adjust the scale factor for the next step
            else:
                # CPU / Apple path, plain 32-bit.
                _, loss = model(x, y)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), tc.grad_clip)
                optimizer.step()

            # 4. Reset gradients. PyTorch ADDS new gradients to old ones, so we must clear them each step.
            optimizer.zero_grad(set_to_none=True)
            losses.append(loss.item())

            if step % 100 == 0:
                # Print the average of the last 100 losses, which is steadier than a single noisy value.
                avg = sum(losses[-100:]) / len(losses[-100:])
                elapsed = time.time() - t0
                print(f"{step:6d} | {lr:10.6f} | {avg:10.4f} | {'--':>10} | {elapsed:7.1f}s")

            if step > 0 and step % tc.eval_interval == 0:
                el = evaluate(model, eval_loader, device)
                avg_train = sum(losses[-tc.eval_interval:]) / min(len(losses), tc.eval_interval)
                elapsed = time.time() - t0
                print(f"{step:6d} | {lr:10.6f} | {avg_train:10.4f} | {el:10.4f} | {elapsed:7.1f}s")

                # Keep the checkpoint with the lowest eval loss. This guards against a later, worse step.
                if el < best_eval:
                    best_eval = el
                    torch.save({
                        "step": step,
                        "model_state_dict": model.state_dict(),   # all learned weights, as a dict of tensors
                        "config": vars(mc),
                        "eval_loss": el,
                    }, os.path.join(tc.output_dir, "best_model.pt"))
                    print(f"  -> Best model (eval={el:.4f})")

            # Periodic snapshots, so a crashed run can resume from the latest one.
            if step > 0 and step % tc.save_interval == 0:
                torch.save({
                    "step": step,
                    "model_state_dict": model.state_dict(),
                    "config": vars(mc),
                }, os.path.join(tc.output_dir, f"step_{step}.pt"))

            step += 1

    # Final save
    torch.save({
        "step": step,
        "model_state_dict": model.state_dict(),
        "config": vars(mc),
        "train_losses": losses,
    }, os.path.join(tc.output_dir, "final_model.pt"))

    elapsed = time.time() - t0
    print(f"\nDone! {elapsed:.0f}s, best eval: {best_eval:.4f}")


if __name__ == "__main__":
    train()

In [ ]:
%%writefile stages.py
"""Training stages for GuppyLM: pretraining first, then SFT and DPO.

Every stage uses the same model and the same training loop (train_stage). What changes from
stage to stage is the DATA and the LOSS FUNCTION:

  Stage 0  pretraining  plain text, predict the next token everywhere      (this file, below)
  Stage 1  SFT          chat text, predict the next token in answers only  (below)
  Stage 2  DPO          pairs of good and bad answers                      (added later)
"""

import json
import os
import random
import time

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer

from config import TrainConfig
from train import get_lr


# ══════════════════════════════════════════════════════════════════════════════
#  STAGE 0: PRETRAINING DATA
# ══════════════════════════════════════════════════════════════════════════════

def write_pretrain_corpus(sentences, train_path, eval_path, eval_fraction=0.02, seed=42):
    """Write a plain-text corpus: one sentence per JSONL line, no chat format.

    Steps, like a real pretraining pipeline in miniature:
      1. remove exact duplicates (repeated text wastes training and leaks into the eval set),
      2. shuffle, so neighbouring sentences are unrelated,
      3. hold out a small part for evaluation.
    """
    unique = sorted(set(s.strip() for s in sentences if s.strip()))  # sorted() first: set order is not stable
    random.Random(seed).shuffle(unique)                              # a fixed seed makes the result repeatable
    n_eval = max(1, int(len(unique) * eval_fraction))
    splits = {eval_path: unique[:n_eval], train_path: unique[n_eval:]}
    for path, items in splits.items():
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
        with open(path, "w") as f:
            for s in items:
                f.write(json.dumps({"text": s}) + "\n")
    return len(unique) - n_eval, n_eval


class PackedDataset(Dataset):
    """Glue all texts into one long stream of tokens, then cut it into equal blocks.

    Real pretraining does the same. It wastes no space on padding, and every block is full.
    Texts are separated by the end-of-text token, so the model learns where a text ends.

      stream:  [the, water, is, warm, <end>, i, love, food, <end>, ...]
      block 0: x = stream[0:128]    y = stream[1:129]    (y is x shifted by one token)
      block 1: x = stream[128:256]  y = stream[129:257]
    """

    def __init__(self, path, tokenizer_path, block_size=128, end_id=2):
        tokenizer = Tokenizer.from_file(tokenizer_path)
        ids = []
        with open(path) as f:
            for line in f:
                ids.extend(tokenizer.encode(json.loads(line)["text"]).ids)
                ids.append(end_id)
        self.tokens = torch.tensor(ids, dtype=torch.long)
        self.block_size = block_size
        # Each block needs block_size + 1 tokens (the extra one is the last target). Leftover tokens are dropped.
        self.n_blocks = (len(ids) - 1) // block_size

    def __len__(self):
        return self.n_blocks

    def __getitem__(self, i):
        start = i * self.block_size
        chunk = self.tokens[start:start + self.block_size + 1]
        return chunk[:-1], chunk[1:]   # all blocks have the same length, so no padding or custom collate is needed


def get_pretrain_loader(path, tokenizer_path, block_size=128, batch_size=32, shuffle=True, end_id=2):
    dataset = PackedDataset(path, tokenizer_path, block_size, end_id)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=shuffle)


# ══════════════════════════════════════════════════════════════════════════════
#  LOSS FUNCTIONS. A loss function takes (model, batch) and returns one number.
# ══════════════════════════════════════════════════════════════════════════════

def lm_loss(model, batch):
    """Plain next-token loss: the model's own cross-entropy, counted at every position."""
    x, y = batch
    _, loss = model(x, y)
    return loss


# ══════════════════════════════════════════════════════════════════════════════
#  STAGE 1: SFT (SUPERVISED FINE-TUNING)
# ══════════════════════════════════════════════════════════════════════════════

IGNORE = -100   # PyTorch's cross_entropy skips every target equal to this value (its default ignore_index)


class SFTDataset(Dataset):
    r"""Chat examples where the loss counts ONLY the assistant's answer.

    Each example is the same ChatML text as before. The difference is in the targets:

      prompt part (targets are -100, no loss):  <|im_start|> user \n are you hungry <|im_end|> \n <|im_start|> assistant \n
      answer part (targets are real tokens):    yes . always yes . <|im_end|>

    We do not teach the model to write the prompt. We only teach it to write the answer.
    The final <|im_end|> is part of the answer, so the model also learns WHEN to stop.
    """

    def __init__(self, pairs, tokenizer_path, max_len=128):
        # pairs: a list of (user_text, reply_text)
        tokenizer = Tokenizer.from_file(tokenizer_path)
        self.samples = []
        for user_text, reply_text in pairs:
            # Prompt and answer are encoded separately and joined. That is exactly what happens at chat time:
            # the prompt is encoded, then the model writes the answer token by token.
            prompt_ids = tokenizer.encode(f"<|im_start|>user\n{user_text}<|im_end|>\n<|im_start|>assistant\n").ids
            reply_ids = tokenizer.encode(f"{reply_text}<|im_end|>").ids
            ids = (prompt_ids + reply_ids)[:max_len]
            x, y = ids[:-1], ids[1:]                  # same one-token shift as always
            # y[i] is the token that follows x[i]. The answer starts at ids[len(prompt_ids)], which is y[len(prompt_ids) - 1].
            n_masked = len(prompt_ids) - 1
            if n_masked >= len(y):
                continue                              # the answer was cut off completely: nothing to learn from
            self.samples.append((x, [IGNORE] * n_masked + y[n_masked:]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        x, y = self.samples[i]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)


def sft_collate(batch):
    # Pad every example to the longest one in the batch. Inputs are padded with <pad> (id 0),
    # targets with IGNORE, so padding never counts in the loss. Because attention is causal,
    # padding at the END cannot change the results for the real tokens before it.
    xs, ys = zip(*batch)
    longest = max(len(x) for x in xs)
    padded_x = torch.zeros((len(xs), longest), dtype=torch.long)
    padded_y = torch.full((len(ys), longest), IGNORE, dtype=torch.long)
    for i, (x, y) in enumerate(zip(xs, ys)):
        padded_x[i, :len(x)] = x
        padded_y[i, :len(y)] = y
    return padded_x, padded_y


def get_sft_loader(pairs, tokenizer_path, max_len=128, batch_size=32, shuffle=True):
    dataset = SFTDataset(pairs, tokenizer_path, max_len)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, collate_fn=sft_collate, drop_last=shuffle)


def sft_loss(model, batch):
    """Next-token loss over the answer tokens only (the targets that are not IGNORE)."""
    x, y = batch
    logits, _ = model(x)                              # no targets passed: we compute the loss ourselves, with the mask
    return F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1), ignore_index=IGNORE)


# ══════════════════════════════════════════════════════════════════════════════
#  THE TRAINING LOOP, SHARED BY ALL STAGES
# ══════════════════════════════════════════════════════════════════════════════

@torch.no_grad()
def evaluate_stage(model, loader, loss_fn, device, max_batches=50):
    model.eval()
    total, n = 0.0, 0
    for batch in loader:
        if n >= max_batches:
            break
        total += loss_fn(model, [t.to(device) for t in batch]).item()
        n += 1
    model.train()
    return total / max(1, n)


def train_stage(model, train_loader, eval_loader, loss_fn, device, save_path,
                max_steps=2000, learning_rate=3e-4, min_lr=3e-5, warmup_steps=100,
                eval_interval=200, log_every=100, weight_decay=0.1, grad_clip=1.0):
    """Train `model` with `loss_fn`. Keep the weights with the lowest eval loss at `save_path`.

    This is the same loop as train.py (forward, backward, clip, update, reset), with the loss
    pulled out into `loss_fn` so that other stages can reuse it. Returns (step_losses, best_eval).
    """
    if len(train_loader) == 0:
        # Without this check, the while loop below would spin forever on an empty loader.
        raise ValueError("train_loader has no batches. Use a smaller batch_size or block_size, or more data.")
    # TrainConfig is reused only as a bag of numbers for get_lr (warmup, cosine decay).
    tc = TrainConfig(max_steps=max_steps, learning_rate=learning_rate, min_lr=min_lr, warmup_steps=warmup_steps)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay, betas=(0.9, 0.95))
    use_amp = device.type == "cuda"                              # mixed precision, as in train.py
    scaler = torch.amp.GradScaler("cuda") if use_amp else None
    os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)

    model.train()
    step, best_eval, losses, t0 = 0, float("inf"), [], time.time()
    print(f"{'Step':>6} | {'LR':>10} | {'Train':>10} | {'Eval':>10} | {'Time':>8}")
    print("-" * 56)

    while step < max_steps:
        for batch in train_loader:
            if step >= max_steps:
                break
            batch = [t.to(device) for t in batch]
            lr = get_lr(step, tc)
            for group in optimizer.param_groups:
                group["lr"] = lr

            if use_amp:
                with torch.amp.autocast("cuda"):
                    loss = loss_fn(model, batch)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss = loss_fn(model, batch)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            losses.append(loss.item())

            if step % log_every == 0:
                avg = sum(losses[-log_every:]) / len(losses[-log_every:])
                print(f"{step:6d} | {lr:10.6f} | {avg:10.4f} | {'--':>10} | {time.time() - t0:7.1f}s")

            if step > 0 and step % eval_interval == 0:
                eval_loss = evaluate_stage(model, eval_loader, loss_fn, device)
                avg = sum(losses[-eval_interval:]) / min(len(losses), eval_interval)
                print(f"{step:6d} | {lr:10.6f} | {avg:10.4f} | {eval_loss:10.4f} | {time.time() - t0:7.1f}s")
                if eval_loss < best_eval:
                    best_eval = eval_loss
                    torch.save({
                        "step": step,
                        "model_state_dict": model.state_dict(),
                        "config": vars(model.config),
                        "eval_loss": eval_loss,
                    }, save_path)
                    print(f"  -> Best model (eval={eval_loss:.4f})")
            step += 1

    print(f"\nDone! {time.time() - t0:.0f}s, best eval: {best_eval:.4f}")
    return losses, best_eval

In [ ]:
%%writefile inference.py
"""GuppyLM inference — simple chat."""

import json
import os
import time
import uuid

import torch
from tokenizers import Tokenizer

from config import GuppyConfig
from model import GuppyLM


class GuppyInference:
    def __init__(self, checkpoint_path, tokenizer_path, device="cpu"):
        self.device = torch.device(device)
        self.tokenizer = Tokenizer.from_file(tokenizer_path)

        # A checkpoint is a pickled Python dict saved by train.py. map_location puts the tensors on our device.
        # weights_only=False allows non-tensor data (the config dict) inside, so only load files you trust.
        ckpt = torch.load(checkpoint_path, map_location=self.device, weights_only=False)

        # Load config.json from same directory as the model file
        config_dir = os.path.dirname(os.path.abspath(checkpoint_path))
        config_path = os.path.join(config_dir, "config.json")

        # Extract state_dict — handle both legacy and standard formats
        # (a state_dict maps each layer name to its weight tensor)
        if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
            state_dict = ckpt["model_state_dict"]
        else:
            state_dict = ckpt

        # Load config — try config.json first, fall back to embedded config
        # We must rebuild a model of exactly the same shape as the one that was trained, or loading fails.
        if os.path.exists(config_path):
            with open(config_path) as f:
                cfg = json.load(f)
            # Support both HF standard keys and our own keys
            # (cfg.get(a, cfg.get(b, default)) means: try key a, then key b, then the default)
            self.config = GuppyConfig(
                vocab_size=cfg.get("vocab_size", 4096),
                max_seq_len=cfg.get("max_position_embeddings", cfg.get("max_seq_len", 128)),
                d_model=cfg.get("hidden_size", cfg.get("d_model", 384)),
                n_layers=cfg.get("num_hidden_layers", cfg.get("n_layers", 6)),
                n_heads=cfg.get("num_attention_heads", cfg.get("n_heads", 6)),
                ffn_hidden=cfg.get("intermediate_size", cfg.get("ffn_hidden", 768)),
                dropout=cfg.get("hidden_dropout_prob", cfg.get("dropout", 0.1)),
                pad_id=cfg.get("pad_token_id", cfg.get("pad_id", 0)),
                bos_id=cfg.get("bos_token_id", cfg.get("bos_id", 1)),
                eos_id=cfg.get("eos_token_id", cfg.get("eos_id", 2)),
            )
        elif isinstance(ckpt, dict) and "config" in ckpt:
            # Keep only keys that GuppyConfig actually has, in case the file holds extra ones.
            valid_fields = {f.name for f in GuppyConfig.__dataclass_fields__.values()}
            self.config = GuppyConfig(**{k: v for k, v in ckpt["config"].items() if k in valid_fields})
        else:
            print("Warning: No config found, using defaults")
            self.config = GuppyConfig()

        # Build an empty model, then copy the trained weights into it.
        self.model = GuppyLM(self.config).to(self.device)
        filtered = {k: v for k, v in state_dict.items() if k in self.model.state_dict()}
        self.model.load_state_dict(filtered)
        self.model.eval()   # inference mode: dropout off

        total, _ = self.model.param_count()
        print(f"GuppyLM loaded: {total/1e6:.1f}M params")

    def chat_completion(self, messages, temperature=0.7, max_tokens=64,
                        top_k=50, **kwargs):
        """Chat completion — takes messages, returns response."""
        # Same steps as the training data: text -> ChatML prompt -> token ids.
        prompt = self._format_prompt(messages)
        input_ids = self.tokenizer.encode(prompt).ids
        prompt_tokens = len(input_ids)
        # The extra [ ] adds the batch dimension: shape (1, T).
        input_t = torch.tensor([input_ids], dtype=torch.long, device=self.device)

        # generate() returns the prompt plus the new tokens. We slice off the prompt part with [prompt_tokens:].
        output_t, _ = self.model.generate(input_t, max_tokens, temperature, top_k)
        output_text = self.tokenizer.decode(output_t[0].tolist()[prompt_tokens:])
        # Truncate at first <|im_end|> — don't let the model leak into the next turn
        if "<|im_end|>" in output_text:
            output_text = output_text.split("<|im_end|>")[0]
        # Also strip any <|im_start|> fragments
        if "<|im_start|>" in output_text:
            output_text = output_text.split("<|im_start|>")[0]
        resp_text = output_text.strip()

        # The return shape copies the OpenAI chat API, so code written for that API works here.
        return {
            "choices": [{
                "message": {"role": "assistant", "content": resp_text},
            }],
        }

    def _format_prompt(self, messages):
        # Build the same ChatML text the model saw in training. The prompt ends with an open
        # "<|im_start|>assistant" line, so the model's natural next step is to write the answer.
        parts = []
        for msg in messages:
            role = msg.get("role", "user")
            content = msg.get("content") or ""
            parts.append(f"<|im_start|>{role}\n{content}<|im_end|>")
        parts.append("<|im_start|>assistant\n")
        return "\n".join(parts)


def main():
    # Command-line chat. Example: python -m guppylm.inference -p "hi guppy"
    import argparse
    p = argparse.ArgumentParser(description="Chat with Guppy")
    p.add_argument("--checkpoint", default="checkpoints/best_model.pt")
    p.add_argument("--tokenizer", default="data/tokenizer.json")
    p.add_argument("--device", default="cpu")
    p.add_argument("--prompt", "-p", help="Single prompt mode: ask one question and exit")
    args = p.parse_args()

    engine = GuppyInference(args.checkpoint, args.tokenizer, args.device)

    if args.prompt:
        result = engine.chat_completion([{"role": "user", "content": args.prompt}])
        print(result["choices"][0]["message"]["content"])
        return

    print("\nGuppy Chat (type 'quit' to exit)")
    while True:
        inp = input("\nYou> ").strip()
        if inp.lower() in ("quit", "exit", "q"):
            break
        result = engine.chat_completion([{"role": "user", "content": inp}])
        msg = result["choices"][0]["message"]
        if msg.get("content"):
            print(f"Guppy> {msg['content']}")


if __name__ == "__main__":
    main()

## 3. Data for stage 0: plain sentences

We reuse the 57,000 fish conversations, but we **take them apart**. Every user sentence and every Guppy sentence becomes a separate piece of plain text. There is no chat format and no pairing of a question with its answer.

This is the point of pretraining: the model reads lots of text and learns how the language works. It does not learn yet that a question should get an answer. That comes from SFT.

We also do three things real pipelines do:
1. **Deduplicate.** Exact repeats waste training time. They would also leak between the training and evaluation sets.
2. **Shuffle**, so neighbouring sentences have nothing to do with each other.
3. **Hold out 2%** for evaluation.

In [ ]:
from datasets import load_dataset
from stages import write_pretrain_corpus

ds = load_dataset('arman-bd/guppylm-60k-generic')
rows = ds['train']

# One list with every user sentence and every Guppy sentence, in no particular order.
sentences = [r['input'] for r in rows] + [r['output'] for r in rows]
print(f'{len(sentences):,} sentences before removing duplicates')

n_train, n_eval = write_pretrain_corpus(sentences, 'data/pretrain_train.jsonl', 'data/pretrain_eval.jsonl')
print(f'{n_train + n_eval:,} unique sentences: {n_train:,} for training, {n_eval:,} held out for evaluation')

print('\nFirst five lines of the corpus (note: unrelated to each other):')
with open('data/pretrain_train.jsonl') as f:
    for _, line in zip(range(5), f):
        print('  ', json.loads(line)['text'])

## 4. The tokenizer (one tokenizer for all stages)

All stages must use the **same** tokenizer, because the model's weights only make sense for the token ids it was trained with. So we train the tokenizer once, now, on text in chat format (the format stages 1 and 2 will use), and reuse it everywhere. Chat markers like `<|im_start|>` become single tokens.

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, processors

# The chat-format text is used ONLY to train the tokenizer here.
chat_texts = [
    f'<|im_start|>user\n{r["input"]}<|im_end|>\n<|im_start|>assistant\n{r["output"]}<|im_end|>'
    for r in rows
]

tokenizer = Tokenizer(models.BPE())
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()
trainer = trainers.BpeTrainer(
    vocab_size=4096,
    special_tokens=['<pad>', '<|im_start|>', '<|im_end|>'],   # ids 0, 1, 2
    min_frequency=2,
    show_progress=False,
)
tokenizer.train_from_iterator(chat_texts, trainer)
tokenizer.post_processor = processors.ByteLevel(trim_offsets=False)
tokenizer.save('data/tokenizer.json')
print(f'Vocabulary size: {tokenizer.get_vocab_size()}')

## 5. Packing text into blocks

The first notebook padded short sentences up to the longest one in each batch. Pretraining does it differently: it **glues all texts into one long stream** and cuts the stream into blocks of exactly 128 tokens. Nothing is wasted on padding.

Texts are separated by the end-of-text token (id 2, the same id as `<|im_end|>`). Here it simply means "this text is over", and the model learns to write it when a sentence is finished.

Let's look at one block. The `y` row is the `x` row shifted by one token, which is the whole training task: **predict the next token at every position**.

In [ ]:
from stages import PackedDataset, get_pretrain_loader

BLOCK = 128
train_ds = PackedDataset('data/pretrain_train.jsonl', 'data/tokenizer.json', block_size=BLOCK)
print(f'{len(train_ds.tokens):,} tokens in the stream -> {len(train_ds):,} blocks of {BLOCK} tokens')

x, y = train_ds[0]
print('x[:12]:', x[:12].tolist())
print('y[:12]:', y[:12].tolist(), ' <- same numbers, one step to the left')

print('\nBlock 0 decoded (id 2 marks where one text ends and the next begins):')
text = tokenizer.decode(x[:60].tolist(), skip_special_tokens=False)
print(text.replace('<|im_end|>', ' ⏎ '))

## 6. Pretrain

We use the same GuppyLM as before, built from scratch (random weights). Before training, the loss should be about `ln(4096) = 8.32`, the loss of a random guess.

`train_stage` is the loop from the first notebook with one change: the loss is passed in as a function (`lm_loss`). Stages 1 and 2 will pass in different loss functions and reuse the rest.

We train for 2,000 steps. The corpus is small (about 218,000 tokens, 53 batches per epoch), so 2,000 steps means the model reads it roughly 38 times. That is plenty, and in fact more than it needs, as the next cell shows.

In [ ]:
from config import GuppyConfig
from model import GuppyLM
from stages import lm_loss, train_stage

config = GuppyConfig()
torch.manual_seed(42)
model = GuppyLM(config).to(device)
print(model.param_summary())

train_loader = get_pretrain_loader('data/pretrain_train.jsonl', 'data/tokenizer.json', BLOCK, batch_size=32, shuffle=True)
eval_loader = get_pretrain_loader('data/pretrain_eval.jsonl', 'data/tokenizer.json', BLOCK, batch_size=32, shuffle=False)
print(f'{len(train_loader)} training batches per epoch, {len(eval_loader)} evaluation batches')

model.eval()
with torch.no_grad():
    batch = [t.to(device) for t in next(iter(train_loader))]
    print(f'Loss before training: {lm_loss(model, batch).item():.3f}   (ln(4096) = 8.318)')

In [ ]:
t0 = time.time()
losses, best_eval = train_stage(
    model, train_loader, eval_loader, lm_loss, device,
    save_path='checkpoints/pretrain/best.pt',
    max_steps=2000, learning_rate=3e-4, warmup_steps=100,
)
print(f'Total wall time: {time.time() - t0:.0f} s')

In [ ]:
import matplotlib.pyplot as plt

def smooth(values, k=50):
    return [sum(values[max(0, i - k + 1): i + 1]) / len(values[max(0, i - k + 1): i + 1]) for i in range(len(values))]

plt.figure(figsize=(7, 3.5))
plt.plot(losses, alpha=0.25, label='per step')
plt.plot(smooth(losses), label='average of last 50 steps')
plt.xlabel('step'); plt.ylabel('training loss'); plt.legend(); plt.title('Stage 0: pretraining loss')
plt.show()
print(f'First loss: {losses[0]:.3f}   Last loss: {losses[-1]:.3f}   Best eval loss: {best_eval:.3f}')

**Read the table above, not only the plot.** The training loss keeps falling to the end. The eval loss, measured on sentences the model never saw, tells a different story: it reaches its lowest value around step 800, then climbs back up. That is **overfitting**: the model starts to memorise its training sentences instead of learning the language.

This is why `train_stage` saves the checkpoint with the best eval loss and not the last one. The file `checkpoints/pretrain/best.pt` holds the model from around step 800.

Two caveats. The evaluation set is small (a few hundred sentences), so the eval numbers are noisy. And the corpus is tiny and very repetitive: only about a tenth of the 114,000 sentences were unique. Real pretraining data is far larger and far more varied, so it takes much longer to overfit.

## 7. What did pretraining teach?

Let's load the best checkpoint and see what the model does. A pretrained model is a **text continuer**: you give it the start of a text and it writes what probably comes next. It has never seen a question followed by an answer, so we expect it to continue sentences, not to answer them.

In [ ]:
ckpt = torch.load('checkpoints/pretrain/best.pt', map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"Loaded the checkpoint from step {ckpt['step']} (eval loss {ckpt['eval_loss']:.3f})")

def complete(text, max_new_tokens=30, temperature=0.8):
    # Encode the text, let the model add tokens, decode everything back to text.
    ids = tokenizer.encode(text).ids
    x = torch.tensor([ids], device=device)
    out, _ = model.generate(x, max_new_tokens=max_new_tokens, temperature=temperature, top_k=50)
    return tokenizer.decode(out[0].tolist())

In [ ]:
print('--- Continuing a sentence ---')
for start in ['the water is', 'i love', 'my favorite thing', 'it is']:
    print(f'{start!r:>22} -> {complete(start)}')

print('\n--- Asking a question in chat format ---')
question = '<|im_start|>user\nare you hungry<|im_end|>\n<|im_start|>assistant\n'
for _ in range(3):
    print(complete(question, max_new_tokens=25).replace('\n', ' | '))

**What to look for.** The continuations should read like fish talk: short, lowercase, about water and food. That is what pretraining gives: the voice and the vocabulary.
The chat-format question is a different story. The model has not been taught to answer. In stage 1 (SFT) it will be.

**Where we are.** The pretrained weights are saved in `checkpoints/pretrain/best.pt`. Stage 1 starts from this file instead of from random weights. That is the key idea: **each stage starts where the previous one stopped**.

## 8. Stage 1: SFT (supervised fine-tuning)

Now we teach the model to **answer**. We use real question-and-answer conversations in chat format, and we start from the pretrained weights instead of from scratch.

One idea makes SFT different from pretraining: the **loss mask**. The model reads the whole conversation, but we only compute the loss on the assistant's answer. The user's question is context, not something the model should learn to write.
PyTorch has a built-in way to do this: any target equal to `-100` is skipped by `cross_entropy`.

The final `<|im_end|>` counts as part of the answer, so the model also learns **when to stop**.

First we build the (question, answer) pairs. The HuggingFace test split (3,000 conversations) becomes our evaluation set.

In [ ]:
from stages import SFTDataset, get_sft_loader, sft_loss, IGNORE

train_pairs = [(r['input'], r['output']) for r in ds['train']]
eval_pairs = [(r['input'], r['output']) for r in ds['test']]
print(f'{len(train_pairs):,} training pairs, {len(eval_pairs):,} evaluation pairs')

# Look at one example: which tokens will the loss count?
sample = SFTDataset(train_pairs[:1], 'data/tokenizer.json')
x, y = sample[0]
print(f'\n{"input token":>16}   {"target"}')
for tok_id, target in list(zip(x.tolist(), y.tolist()))[:40]:
    shown = tokenizer.decode([tok_id], skip_special_tokens=False).replace('\n', '\\n')
    goal = 'masked (no loss)' if target == IGNORE else repr(tokenizer.decode([target], skip_special_tokens=False))
    print(f'{shown!r:>16}   {goal}')

Read the table from top to bottom. While the input is still the user's question, the target is masked. Once the input reaches `assistant` and the newline after it, the targets become the answer tokens, one step ahead of the input.

Before we train, let's measure the **answer loss** of the pretrained model on the evaluation conversations: how surprised is it by the real answers when it is given the question in chat format? We will measure again after SFT.

In [ ]:
from stages import evaluate_stage

sft_train_loader = get_sft_loader(train_pairs, 'data/tokenizer.json', batch_size=32, shuffle=True)
sft_eval_loader = get_sft_loader(eval_pairs, 'data/tokenizer.json', batch_size=32, shuffle=False)
print(f'{len(sft_train_loader)} training batches per epoch, {len(sft_eval_loader)} evaluation batches')

# Start from the pretrained checkpoint (the best one from stage 0).
sft_model = GuppyLM(config).to(device)
sft_model.load_state_dict(torch.load('checkpoints/pretrain/best.pt', map_location=device, weights_only=False)['model_state_dict'])

before = evaluate_stage(sft_model, sft_eval_loader, sft_loss, device, max_batches=100)
print(f'Answer loss BEFORE SFT: {before:.3f}   (random guessing would be {torch.log(torch.tensor(4096.0)).item():.3f})')

Now train. Two settings differ from stage 0:

- **A lower learning rate (1e-4 instead of 3e-4).** The model already knows the language. We want to adjust it, not overwrite what it learned.
- **Fewer steps.** 1,500 steps of 32 conversations is less than one pass over the 57,000 training pairs, so overfitting is not a worry this time.

In [ ]:
t0 = time.time()
sft_losses, sft_best = train_stage(
    sft_model, sft_train_loader, sft_eval_loader, sft_loss, device,
    save_path='checkpoints/sft/best.pt',
    max_steps=1500, learning_rate=1e-4, min_lr=1e-5, warmup_steps=50, eval_interval=100,
)
print(f'Total wall time: {time.time() - t0:.0f} s')

In [ ]:
plt.figure(figsize=(7, 3.5))
plt.plot(sft_losses, alpha=0.25, label='per step')
plt.plot(smooth(sft_losses), label='average of last 50 steps')
plt.xlabel('step'); plt.ylabel('answer loss'); plt.legend(); plt.title('Stage 1: SFT loss')
plt.show()
print(f'Answer loss on the evaluation set:  before SFT {before:.3f}  ->  after SFT {sft_best:.3f}')

Compare the two numbers: the answer loss on the held-out conversations before and after SFT. Then look at the `Eval` column in the training table above. If it was still falling at the last row, the model had not finished learning, and more steps would help.

## 9. Pretrained versus SFT: same questions, two models

We load both saved checkpoints with the same inference code and ask them the same questions in chat format.

In [ ]:
from inference import GuppyInference

engines = {
    '0 pretrained': GuppyInference('checkpoints/pretrain/best.pt', 'data/tokenizer.json', device=str(device)),
    '1 SFT':        GuppyInference('checkpoints/sft/best.pt', 'data/tokenizer.json', device=str(device)),
}

def reply(engine, text, temperature=0.7):
    r = engine.chat_completion([{'role': 'user', 'content': text}], temperature=temperature, max_tokens=64)
    return r['choices'][0]['message']['content']

torch.manual_seed(1)
for q in ['hi guppy', 'are you hungry', 'do you like bubbles', 'what is the meaning of life', 'tell me a joke', 'goodnight guppy']:
    print(f'You> {q}')
    for name, engine in engines.items():
        print(f'   {name:>13}: {reply(engine, q)!r}')
    print()

### Chat with either stage

Pick a stage in the dropdown, type a message, and press **Enter**. Switch back and forth to feel the difference between the stages.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

stage_menu = widgets.Dropdown(options=list(engines), value='1 SFT', description='stage', style={'description_width': 'initial'})
box = widgets.Text(placeholder='say something to Guppy, then press Enter', layout=widgets.Layout(width='520px'))
log = widgets.Output()

def on_send(_):
    text = box.value.strip()
    if not text:
        return
    box.value = ''
    with log:
        print(f'You>   {text}')
        print(f'Guppy ({stage_menu.value}) > {reply(engines[stage_menu.value], text)}\n')

box.on_submit(on_send)
display(stage_menu, box, log)

## Optional: run the unit tests

The repository has tests for `stages.py`: duplicate removal, block packing, and a short training run on a toy corpus. This cell downloads the repository and runs them.

In [ ]:
!rm -rf /content/repo && git clone -q -b feature/beginner-notebook https://github.com/i3wangyi/guppylm /content/repo
!cd /content/repo && python -m unittest discover -s tests -v 2>&1 | grep -E "^(test_|Ran|OK|FAILED)"